---
title: "4.4 Robust and clustered standard errors"
---

::: {.lesson-tools}
[Open in Colab](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/lessons/04-regression/04-robust-clustered.ipynb){.btn .btn-primary}
[Download notebook](../../downloads/04-regression/04-robust-clustered.ipynb){.btn .btn-outline-secondary}
:::

**Run in your browser.** Colab needs a Google account to run and save work. If its GitHub link cannot open, download this notebook and use **File → Upload notebook** at [Colab](https://colab.research.google.com/). You can read the website without an account. A notebook is a document containing explanatory text and runnable code cells.

**Time:** about 40 minutes, including exercises · **Before you begin:** Complete [4.3 Multiple regression, categories and interactions](../04-regression/03-multiple-regression.ipynb) first. Each notebook starts afresh: it does not remember variables from another lesson.

## You will learn

- Separate coefficient estimation from the uncertainty calculation.
- Choose HC1 or country-clustered standard errors for a stated dependence assumption.
- Check the analysis sample, group alignment and number of clusters.

## Why it matters

A regression's **standard errors** quantify coefficient uncertainty under assumptions about errors. **Heteroskedasticity** means the error variance can differ across observations. **Clustering** allows errors within a chosen group to be related, while relying on assumptions about independence across groups and enough groups for the approximation. These are uncertainty choices, not repairs for an omitted cause, a wrong functional form or biased data. We start with made-up numbers, then use the repeated country panel.

## Worked example

### Prepare the analysis libraries

Run this prepared cell **before importing other analysis libraries**. It uses the versions generated from the course's single dependency file. If needed, it installs free packages inside this temporary Python runtime, not on your own computer. An internet connection is needed for installation. If it reports that a restart is required, restart the Colab session and run from the top; mixing already loaded old code with newly installed libraries can produce misleading errors.

In [ ]:
#| code-fold: true
#| code-summary: "Prepared analysis-library setup (run first)"
# Generated from requirements.txt; installs only in this Python runtime.
import importlib.metadata
from pathlib import Path
import subprocess
import sys

requested = ['statsmodels==0.15.0']
needs_install = False
for requirement in requested:
    name, version = requirement.split("==")
    try:
        needs_install = needs_install or importlib.metadata.version(name) != version
    except importlib.metadata.PackageNotFoundError:
        needs_install = True
if needs_install:
    loaded = any(name in sys.modules for name in ["numpy", "pandas", "scipy", "statsmodels", "linearmodels"])
    constraints = Path("scratch/course-analysis-constraints.txt")
    constraints.parent.mkdir(parents=True, exist_ok=True)
    constraints.write_text('# Python 3.12; tested on macOS. CI also installs this file on Ubuntu.\n# Direct dependencies and their resolved transitive dependencies are pinned.\n# Add a library only when an available lesson or a course check needs it.\nappnope==1.0.0; sys_platform == "darwin"\nasttokens==3.0.2\nattrs==26.1.0\ncomm==0.2.3\ncontourpy==1.4.0\ncycler==0.12.1\ndebugpy==1.8.22\net_xmlfile==2.0.0\nexecuting==2.3.0\nfastjsonschema==2.22.3\nfonttools==4.68.0\nipykernel==7.4.0\nipython==9.18.1\nipython_pygments_lexers==1.1.1\njedi==0.20.1\njsonschema==4.26.0\njsonschema-specifications==2025.9.1\njupyter_client==8.10.0\njupyter_core==5.9.1\nkiwisolver==1.5.1\nmatplotlib==3.11.2\nmatplotlib-inline==0.2.2\nnbclient==0.11.0\nnbformat==5.11.1\nnest-asyncio2==1.7.4\nnumpy==2.5.4\nopenpyxl==3.1.5\npackaging==26.3\npandas==3.1.1\nparso==0.8.7\npexpect==4.9.0; sys_platform != "win32"\npillow==13.0.0\nplatformdirs==4.14.5\nprompt_toolkit==3.0.53\npsutil==7.2.2\nptyprocess==0.7.0; sys_platform != "win32"\npure_eval==0.2.4\nPygments==2.21.0\npyparsing==3.3.3\npython-dateutil==4.0.0\nPyYAML==6.0.3\npyzmq==27.3.0\nreferencing==0.37.0\nrpds-py==2026.9.1\nstack-data==0.6.3\ntornado==6.6.1\ntraitlets==5.16.1\ntyping_extensions==4.16.0\ntzdata==2026.6\nwcwidth==0.9.4\n\n# Statistics and fixed-effects models, with their newly required dependencies.\nformulaic==1.2.2\ninterface_meta==2.0.1\nlinearmodels==7.1\nmypy_extensions==1.2.0\nnarwhals==2.28.0\npatsy==1.0.3\npyhdfe==0.2.1\nscipy==1.18.1\nstatsmodels==0.15.0\nwrapt==2.5.2\n', encoding='utf-8')
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--constraint", str(constraints), *requested], check=True)
    if loaded:
        raise RuntimeError("Packages installed. Restart the notebook session, then run all cells from the top so updated libraries are used together.")
print("Analysis libraries ready:", ", ".join(requested))

Look for **Analysis libraries ready**. In the pinned course environment this cell installs nothing. Colab may take a few minutes the first time. The current course environment uses Python 3.12; an older runtime that cannot install the pinned dependencies needs a current Python runtime rather than arbitrary version substitutions.

### See coefficients and uncertainty as separate outputs

Fit the same invented measurements twice, once with conventional uncertainty and once with HC1. Both use OLS for the coefficient estimates. The tiny made-up table is for commands only; it does not justify real-world inference.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
small = pd.DataFrame({
    "country": ["Northland", "Northland", "Eastria", "Eastria", "Southport", "Southport"],
    "year": [2000, 2020, 2000, 2020, 2000, 2020],
    "gdp_pc": [8000, 15000, 21000, 30000, 2500, 4200],
    "life_exp": [68.0, 73.5, 76.0, 80.1, 58.2, 64.0],
})
small["log_gdp_pc"] = np.log(small["gdp_pc"])
toy_specification = smf.ols("life_exp ~ log_gdp_pc", data=small)
toy_conventional = toy_specification.fit()
toy_hc1 = toy_specification.fit(cov_type="HC1", use_t=True)
print(pd.DataFrame({"coefficient": toy_hc1.params, "conventional_se": toy_conventional.bse, "HC1_se": toy_hc1.bse}).round(3))

Changing to HC1 leaves the fitted coefficients unchanged but can change their standard errors. HC1 includes a degrees-of-freedom scale correction, where degrees of freedom account for the parameters estimated from the sample. Robust standard errors can be larger **or smaller** than conventional ones.

### Prepare the real country panel

Run this prepared setup once in a fresh session. It uses the course CSV in a local checkout, or downloads its fixed, checksum-verified public source in Colab. A checksum is a fingerprint of a file's contents; a mismatch stops the download from being used. You do not need to retype the setup. It does not use your Google Drive or any private data.

In [ ]:
# Prepared setup: use the course CSV locally; fetch its pinned source in Colab.
from pathlib import Path
import hashlib
import io
from urllib.request import urlopen
import pandas as pd

data_file = Path("data/gapminder.csv")
if not data_file.exists():
    source = "https://raw.githubusercontent.com/jennybc/gapminder/391b5a40d3574c3b89d889dd7d72f7f3fdb4a70c/inst/extdata/gapminder.tsv"
    with urlopen(source, timeout=30) as response:
        raw = response.read()
    expected = "f74275b4a2f4a7c8e3a09a752f9d6e43d1759d78d5cbfdebaaa3e1a6efc68f78"
    if hashlib.sha256(raw).hexdigest() != expected:
        raise ValueError("The data download changed. Please report this to the course maintainer.")
    downloaded = pd.read_csv(io.BytesIO(raw), sep="\t", dtype=str)
    downloaded = downloaded.rename(columns={"continent": "region", "lifeExp": "life_exp", "pop": "population", "gdpPercap": "gdp_pc"})
    data_file.parent.mkdir(parents=True, exist_ok=True)
    downloaded.to_csv(data_file, index=False, lineterminator="\n")
print("Course data ready: data/gapminder.csv")

Look for **Course data ready**. Colab working files are temporary; rerun this cell after a runtime reset. The data cover 142 countries at five-year intervals from 1952 to 2007. Each row is one country in one year.

### Preserve the country grouping vector

The full panel repeats 142 countries over 12 observed years. Prepare complete cases first, then obtain the cluster labels from exactly that prepared DataFrame. Country clustering addresses possible within-country dependence across time; it does not treat all 1704 country-years as independent units for uncertainty.

In [ ]:
countries = pd.read_csv(data_file)
sample = countries[["country", "year", "life_exp", "gdp_pc"]].dropna().copy()
sample["log_gdp_pc"] = np.log(sample["gdp_pc"])
print("Country-years:", len(sample))
print("Countries:", sample["country"].nunique())
print(sample.groupby("country")["year"].size().value_counts())

The counts are 1704 country-years, 142 countries and 12 observations in each country. Constructing group labels before dropping observations can misalign them with the rows used by a model. With a different dataset, report both sample size and cluster count after all sample restrictions.

### Hold the formula fixed while comparing uncertainty

Fit a pooled model with year categories to account for shared fitted year-level differences. This is not yet a country fixed-effects model. For country clustering, provide `groups`, request a finite-sample scale correction, and retain the group-based degrees-of-freedom correction. Explicit use_t=True requests t-based approximate intervals.

In [ ]:
specification = smf.ols("life_exp ~ log_gdp_pc + C(year)", data=sample, missing="raise")
conventional = specification.fit()
hc1 = specification.fit(cov_type="HC1", use_t=True)
clustered = specification.fit(
    cov_type="cluster", use_t=True,
    cov_kwds={"groups": sample["country"], "use_correction": True, "df_correction": True},
)
comparison = pd.DataFrame({
    "coefficient": clustered.params,
    "conventional_se": conventional.bse,
    "HC1_se": hc1.bse,
    "country_clustered_se": clustered.bse,
})
comparison.loc[["log_gdp_pc"]].round(4)

The income coefficient agrees across all three fits. The three uncertainty columns use different dependence assumptions. HC1 alone does not allow serial dependence within a country. Clustering by the five broad regions would have very few clusters and is not a substitute for the country dependence structure. Never choose a covariance setting just because it gives a smaller p-value.

### Read the chosen interval and cluster count

Inspect the t-based country-clustered interval. statsmodels stores the group-corrected inference degrees of freedom separately from the ordinary residual degrees of freedom. Here 142 country clusters give 141 inference degrees of freedom.

In [ ]:
print("Covariance type:", clustered.cov_type)
print("Inference degrees of freedom:", clustered.df_resid_inference)
print(clustered.conf_int().loc[["log_gdp_pc"]].round(4))
print("Same point estimates:", np.allclose(conventional.params, clustered.params))

Country clustering permits dependence within a country but does not automatically handle arbitrary dependence between countries. Common year controls address shared fitted levels, not every cross-country correlated error. Few or highly unbalanced clusters can make standard approximations unreliable. The historical extract's sampling and measurement limitations remain regardless of how standard errors are computed.

## Try it

Work in your own notebook copy. Add a new code cell for each task, try it yourself, then open the solution to compare. Run from the top if a name is missing.

### 1. Show a cross-section uncertainty setting

Fit the same simple 2007 model with HC1 and use_t=True. Report N and the income standard error. Why is country clustering not adding repeated-time information in that single-year sample?

<details><summary>Solution 1</summary>

Run this complete solution after the worked example.

In [ ]:
cross_section = sample.loc[sample["year"] == 2007].copy()
cross_model = smf.ols("life_exp ~ log_gdp_pc", data=cross_section, missing="raise").fit(cov_type="HC1", use_t=True)
print(int(cross_model.nobs))
print(cross_model.bse["log_gdp_pc"])

There are 142 observations, one per country, with HC1 standard error about 0.334 for the log-income slope. Every country appears once, so clustering by country cannot use within-country repeated observations there; its relationship to robust covariance also depends on correction conventions.

</details>

### 2. Verify the common sample

Print the number of observations and income coefficient for all three full-panel fits. Do not compare uncertainty as if sample changes were harmless.

<details><summary>Solution 2</summary>

Run this complete solution after the worked example.

In [ ]:
for label, result in [("Conventional", conventional), ("HC1", hc1), ("Country clustered", clustered)]:
    print(label, int(result.nobs), round(result.params["log_gdp_pc"], 6))

All three use 1704 rows and the same income coefficient. A coefficient difference would mean something beyond this covariance change had changed, such as sample or specification.

</details>

### 3. Count clusters after a restriction

Restrict the panel to years 1982 onward. Recompute the number of country-years and countries from that restricted sample.

<details><summary>Solution 3</summary>

Run this complete solution after the worked example.

In [ ]:
recent_panel = sample.loc[sample["year"] >= 1982].copy()
print(len(recent_panel))
print(recent_panel["country"].nunique())

The restriction leaves 852 country-years and 142 countries. A shorter time window does not necessarily mean fewer clusters; check the actual data rather than deriving a cluster count from row count alone.

</details>

## When it goes wrong

The following examples deliberately trigger specific errors. The prepared `try`/`except` wrapper catches that named error and prints its message so the notebook can continue. It is for these demonstrations; an unexpected error elsewhere should still stop execution. You can read the message without memorizing the wrapper.

### The covariance label is not supported

A generic word may not be an accepted API option.

In [ ]:
try:
    specification.fit(cov_type="robust_typo")
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

Use a documented choice such as HC1 or cluster and state its assumptions. The word “robust” does not identify one universal method.

### Cluster labels have the wrong length

Drop one group label without dropping the corresponding model row, deliberately creating a mismatch.

In [ ]:
try:
    specification.fit(cov_type="cluster", cov_kwds={"groups": sample["country"].iloc[:-1]})
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

Build group labels from the final analysis sample in the same order. Matching lengths alone are not enough if you separately reorder one vector; preserve the shared row selection.

## Recap

- The covariance choice changes uncertainty, not OLS point estimates on a fixed model/sample.
- HC1 addresses unequal error variances, while clustering can allow within-group dependence.
- Check final sample alignment, cluster count and correction conventions.
- Neither covariance choice establishes causation or fixes arbitrary cross-cluster dependence.

**Next:** [4.5 Fixed effects and panel models](../04-regression/05-fixed-effects.ipynb).